# 🎓 Digital Agri xAI: ฝึกสอนโมเดลปัญญาประดิษฐ์เพื่อการเกษตรและสิ่งแวดล้อม
### 🌿 หลักสูตรปฏิบัติการ Low-Code / No-Code AI Workshop
**วิทยากร:** ผศ.ดร.ชีวะ ทัศนา  
**สาขาวิชา:** ปัญญาประดิษฐ์เพื่อเกษตรดิจิทัลและสิ่งแวดล้อม คณะวิทยาศาสตร์และเทคโนโลยี มหาวิทยาลัยราชภัฏรำไพพรรณี

---
> [!TIP]
> **วิธีใช้งานง่ายๆ สำหรับผู้เริ่มต้น (Zero Programming Required):**
> ไม่จำเป็นต้องมีความรู้ด้านการเขียนโปรแกรม เพียงทำตามขั้นตอนด้านล่าง โดยคลิกปุ่ม Play **[ ▶ ]** ในแต่ละข้อตามลำดับ!
> - **ขั้นตอนที่ 1:** ตรวจสอบการเชื่อมต่อและดึงพลัง GPU ประมวลผลบนคลาวด์
> - **ขั้นตอนที่ 2:** เลือกชุดข้อมูลวิจัย หรืออัปโหลดภาพของท่านเอง พร้อม**แสดงภาพตัวอย่างของข้อมูลและระบบตรวจจับวัตถุเป้าหมาย (Intelligent ROI Detection)**
> - **ขั้นตอนเสริม (2.1):** ตรวจสอบภาพตัวอย่างเจาะลึก ทดสอบระบบตรวจจับวัตถุ และวิเคราะห์ความคมชัด/แสงสะท้อน (Visual Gate Telemetry)
> - **ขั้นตอนที่ 3:** ตั้งค่าพารามิเตอร์แบบ Low-Code เริ่มการฝึกสอน และประเมินผลโมเดลเชิงลึก (Deep Evaluation Suite)
> - **ขั้นตอนที่ 4:** ส่งออกโมเดลมาตรฐานเปิด (ONNX) และดาวน์โหลดลงเครื่องเพื่อนำไปใช้งานจริง


In [ ]:
#@title 🚀 ขั้นตอนที่ 1: ตรวจสอบการเชื่อมต่อและดึงพลัง GPU ประมวลผล { display-mode: "form" }
import torch
import sys

print("=" * 65)
print("🌱 มหาวิทยาลัยราชภัฏรำไพพรรณี | Digital Agri xAI Workshop")
print("=" * 65)
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    print(f"✅ ยอดเยี่ยม! ตรวจพบฮาร์ดแวร์เร่งความเร็ว GPU: {device_name}")
    print("⚡ พร้อมสำหรับการฝึกสอนโมเดล AI ด้วยความเร็วสูงสุดบนคลาวด์")
else:
    print("💡 กำลังประมวลผลบน Standard CPU (หากต้องการความเร็วสูงขึ้น แนะนำไปที่เมนู Runtime > Change runtime type > เลือก T4 GPU)")


In [ ]:
#@title 📂 ขั้นตอนที่ 2: เลือกชุดข้อมูลวิจัย หรืออัปโหลดภาพของท่านเอง พร้อมแสดงภาพตัวอย่างและตรวจจับวัตถุ (Data Preview & ROI Detection) { display-mode: "form" }
import os
import shutil
import random
import zipfile
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

#@markdown **เลือกชุดข้อมูลจริงที่ต้องการฝึกอบรม (100% Real Research Data):**
selected_dataset = "1.1 Soil Analysis: จำแนกชนิดเนื้อดิน 5 ชนิด" #@param ["📁 [ของฉันเอง] อัปโหลดไฟล์ภาพของฉันเอง (1-Click Upload .ZIP)", "1.1 Soil Analysis: จำแนกชนิดเนื้อดิน 5 ชนิด", "2.1 Plant Diseases: โรคใบไหม้/ใบจุดทุเรียนจันทบุรี", "2.1 Plant Diseases: โรคราสนิมและใบไหม้ข้าวโพด", "2.1 Plant Diseases: โรคข้าว (โรคไหม้/ขอบใบแห้ง/ใบจุดสีน้ำตาล)", "2.1 Plant Diseases: โรคมะเขือเทศ (ใบไหม้/ใบจุดวงแหวน/รากำมะหยี่)", "2.2 Plant Insects: แมลงศัตรูพืชทุเรียน (เพลี้ยไก่แจ้)", "2.2 Plant Insects: แมลงศัตรูพืชทั่วไป (หนอนกระทู้/หนอนเจาะสมอฝ้าย/หนอนใยผัก)", "2.2 Plant Insects: แมลงศัตรูพืชข้าว (เพลี้ยกระโดด/หนอนกอข้าว/หนอนห่อใบ)"]

#@markdown **การแสดงตัวอย่างข้อมูลและระบบตรวจจับวัตถุ (Data Preview & Intelligent Object Detection):**
show_sample_preview = True #@param {type:"boolean"}
show_roi_detection = True #@param {type:"boolean"}
samples_per_class = 2 #@param [1, 2, 3] {type:"raw"}

target_dir = "dataset"
if os.path.exists(target_dir):
    shutil.rmtree(target_dir)
os.makedirs(f"{target_dir}/train", exist_ok=True)
os.makedirs(f"{target_dir}/val", exist_ok=True)

if "[ของฉันเอง]" in selected_dataset:
    print("📤 ระบบพร้อมรับไฟล์ของคุณแล้ว! กรุณาคลิกปุ่ม 'Choose Files' ด้านล่างเพื่อเลือกไฟล์ .zip จากคอมพิวเตอร์ของคุณ:")
    from google.colab import files
    uploaded = files.upload()
    zip_filename = None
    for fn in uploaded.keys():
        if fn.lower().endswith(".zip"):
            zip_filename = fn
            break
    
    if zip_filename is None:
        raise ValueError("❌ ไม่พบไฟล์ .zip กรุณาบีบอัดโฟลเดอร์ภาพเป็นไฟล์ .zip แล้วลองใหม่อีกครั้ง")
    
    print(f"📦 กำลังแตกไฟล์และจัดสรรข้อมูลอัตโนมัติจาก: {zip_filename}...")
    temp_raw = "custom_raw_unzipped"
    if os.path.exists(temp_raw):
        shutil.rmtree(temp_raw)
    with zipfile.ZipFile(zip_filename, 'r') as zf:
        zf.extractall(temp_raw)
    
    # ค้นหาโฟลเดอร์คลาส
    subdirs = [d for d in os.listdir(temp_raw) if os.path.isdir(os.path.join(temp_raw, d)) and not d.startswith(".") and not d.startswith("__")]
    root_raw = temp_raw
    if len(subdirs) == 1 and ("train" not in subdirs):
        root_raw = os.path.join(temp_raw, subdirs[0])
        subdirs = [d for d in os.listdir(root_raw) if os.path.isdir(os.path.join(root_raw, d)) and not d.startswith(".") and not d.startswith("__")]
    
    if "train" in subdirs and "val" in subdirs:
        shutil.copytree(f"{root_raw}/train", f"{target_dir}/train", dirs_exist_ok=True)
        shutil.copytree(f"{root_raw}/val", f"{target_dir}/val", dirs_exist_ok=True)
        dataset_label = "ชุดข้อมูลของผู้เรียน (ตรวจพบการแบ่ง train/val เรียบร้อย)"
    else:
        print("⚙️ กำลังสุ่มแบ่งข้อมูล Train (80%) และ Validation (20%) อัตโนมัติ...")
        for cls in subdirs:
            cls_path = os.path.join(root_raw, cls)
            images = [f for f in os.listdir(cls_path) if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))]
            if not images:
                continue
            random.seed(42)
            random.shuffle(images)
            split_idx = max(1, int(len(images) * 0.8))
            train_imgs = images[:split_idx]
            val_imgs = images[split_idx:] if len(images) > 1 else images[:1]
            
            os.makedirs(f"{target_dir}/train/{cls}", exist_ok=True)
            os.makedirs(f"{target_dir}/val/{cls}", exist_ok=True)
            for im in train_imgs:
                shutil.copy(os.path.join(cls_path, im), f"{target_dir}/train/{cls}/{im}")
            for im in val_imgs:
                shutil.copy(os.path.join(cls_path, im), f"{target_dir}/val/{cls}/{im}")
        dataset_label = f"ชุดข้อมูลของผู้เรียน ({len(subdirs)} คลาส)"
else:
    repo_dir = "smart-agri-xai-workshop"
    if not os.path.exists(repo_dir):
        print("🚀 กำลังดาวน์โหลดคลังข้อมูลและโค้ดวิจัยจริงจาก GitHub ของ ผศ.ดร.ชีวะ ทัศนา...")
        !git clone --depth 1 https://github.com/Tsanaphy2023/smart-agri-xai-workshop.git
        print("✅ ดาวน์โหลดคลังข้อมูลสำเร็จเรียบร้อย!")
    else:
        print("🔄 กำลังซิงก์อัปเดตชุดข้อมูลภาพจริงล่าสุดจาก GitHub...")
        !git -C smart-agri-xai-workshop pull
        print("✅ ซิงก์คลังข้อมูลภาพจริง 100% เป็นเวอร์ชันล่าสุดเรียบร้อยแล้ว!")
    
    if "1.1 Soil Analysis" in selected_dataset:
        src = f"{repo_dir}/datasets/Soil/SoilType/01_Real_Data"
        dataset_label = "[1.1 Soil Analysis] จำแนกชนิดเนื้อดิน 5 ชนิด (ดินดำ, ดินลูกรัง, ดินเหนียว, ดินพรุ, ดินเหลือง)"
    elif "ทุเรียนจันทบุรี" in selected_dataset and "2.1 Plant Diseases" in selected_dataset:
        src = f"{repo_dir}/datasets/Plant/PlantDiseases/Durian_ทุเรียน/01_Real_Data"
        dataset_label = "[2.1 Plant Diseases] โรคใบจุดและใบไหม้ทุเรียน จันทบุรี (แอนแทรคโนส, จุดสนิมสาหร่าย, ใบจุด, ปกติ)"
    elif "ข้าวโพด" in selected_dataset and "2.1 Plant Diseases" in selected_dataset:
        src = f"{repo_dir}/datasets/Plant/PlantDiseases/Maize_ข้าวโพด/01_Real_Data"
        dataset_label = "[2.1 Plant Diseases] โรคราสนิมและใบไหม้ข้าวโพด (ราสนิม, ใบจุดสีเทา, ใบไหม้, ปกติ)"
    elif "โรคข้าว" in selected_dataset and "2.1 Plant Diseases" in selected_dataset:
        src = f"{repo_dir}/datasets/Plant/PlantDiseases/Rice_ข้าว/01_Real_Data"
        dataset_label = "[2.1 Plant Diseases] โรคข้าว (โรคไหม้, ขอบใบแห้ง, ใบจุดสีน้ำตาล)"
    elif "มะเขือเทศ" in selected_dataset and "2.1 Plant Diseases" in selected_dataset:
        src = f"{repo_dir}/datasets/Plant/PlantDiseases/Tomato_มะเขือเทศ/01_Real_Data"
        dataset_label = "[2.1 Plant Diseases] โรคมะเขือเทศ (ใบไหม้, ใบจุดวงแหวน, รากำมะหยี่, ปกติ)"
    elif "แมลงศัตรูพืชทุเรียน" in selected_dataset:
        src = f"{repo_dir}/datasets/Plant/PlantInsect/Durian_ทุเรียน/01_Real_Data"
        dataset_label = "[2.2 Plant Insects] แมลงศัตรูพืชทุเรียน (เพลี้ยไก่แจ้, ปกติ)"
    elif "แมลงศัตรูพืชข้าว" in selected_dataset:
        src = f"{repo_dir}/datasets/Plant/PlantInsect/Rice_ข้าว/01_Real_Data"
        dataset_label = "[2.2 Plant Insects] แมลงศัตรูพืชข้าว (เพลี้ยกระโดดสีน้ำตาล, หนอนกอข้าว, หนอนห่อใบ, ปกติ)"
    else:
        src = f"{repo_dir}/datasets/Plant/PlantInsect/General_Crops_พืชทั่วไป/01_Real_Data"
        dataset_label = "[2.2 Plant Insects] แมลงศัตรูพืชทั่วไป (หนอนกระทู้, หนอนเจาะสมอฝ้าย, หนอนใยผัก, ปกติ)"
    
    shutil.copytree(f"{src}/train", f"{target_dir}/train", dirs_exist_ok=True)
    shutil.copytree(f"{src}/val", f"{target_dir}/val", dirs_exist_ok=True)

classes = sorted(os.listdir(f"{target_dir}/train"))
train_count = sum(len(files) for _, _, files in os.walk(f"{target_dir}/train"))
val_count = sum(len(files) for _, _, files in os.walk(f"{target_dir}/val"))
print(f"📦 โหลดชุดข้อมูลสำเร็จ: {dataset_label}")
print(f"🎯 พบคลาสข้อมูลทั้งหมด ({len(classes)} คลาส): {classes}")
print(f"📸 สรุปจำนวนภาพจริง: Train = {train_count} ภาพ | Validation = {val_count} ภาพ")
print("✅ ข้อมูลภาพถ่ายจริง 100% พร้อมสำหรับการฝึกสอนโมเดล AI ในขั้นตอนถัดไป!")

# ==============================================================================
# 🖼️ ส่วนแสดงภาพตัวอย่างของข้อมูล พร้อมระบบตรวจจับวัตถุเป้าหมาย (Intelligent ROI Detection)
# ==============================================================================
def detect_agri_roi(cv_bgr, padding_ratio=0.05):
    """ตรวจจับกรอบวัตถุเป้าหมาย (ดิน/ใบพืช/แมลง) ด้วย Saliency & Morphological Gradient"""
    h, w = cv_bgr.shape[:2]
    gray = cv2.cvtColor(cv_bgr, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (7, 7), 0)
    sobelx = cv2.Sobel(blurred, cv2.CV_64F, 1, 0, ksize=3)
    sobely = cv2.Sobel(blurred, cv2.CV_64F, 0, 1, ksize=3)
    grad_mag = np.sqrt(sobelx**2 + sobely**2)
    grad_norm = cv2.normalize(grad_mag, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    _, thresh = cv2.threshold(grad_norm, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (15, 15))
    closed = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel)
    contours, _ = cv2.findContours(closed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    center_x, center_y = w / 2.0, h / 2.0
    best_bbox = (int(w * 0.05), int(h * 0.05), int(w * 0.9), int(h * 0.9))
    best_score = 0.85
    max_val = 0.0
    
    if contours:
        for cnt in contours:
            area = cv2.contourArea(cnt)
            if area < (w * h * 0.03):
                continue
            bx, by, bw, bh = cv2.boundingRect(cnt)
            cnt_cx, cnt_cy = bx + bw / 2.0, by + bh / 2.0
            dist_norm = np.sqrt((cnt_cx - center_x)**2 + (cnt_cy - center_y)**2) / np.sqrt(center_x**2 + center_y**2)
            area_ratio = area / (w * h)
            score = area_ratio * 0.6 + (1.0 - dist_norm) * 0.4
            if score > max_val:
                max_val = score
                pw, ph = int(bw * padding_ratio), int(bh * padding_ratio)
                x1 = max(0, bx - pw)
                y1 = max(0, by - ph)
                x2 = min(w, bx + bw + pw)
                y2 = min(h, by + bh + ph)
                best_bbox = (x1, y1, x2 - x1, y2 - y1)
                best_score = min(0.99, 0.72 + float(score) * 0.27)
                
    annotated = cv_bgr.copy()
    bx, by, bw, bh = best_bbox
    # ตีกรอบสีเขียว Emerald
    cv2.rectangle(annotated, (bx, by), (bx + bw, by + bh), (16, 185, 129), 2)
    # มุมเล็งเป้าไฮเทค (Corner Brackets)
    c_len = min(22, max(6, int(min(bw, bh) * 0.15)))
    for (cx, cy, dx, dy) in [
        (bx, by, 1, 1), (bx + bw, by, -1, 1),
        (bx, by + bh, 1, -1), (bx + bw, by + bh, -1, -1)
    ]:
        cv2.line(annotated, (cx, cy), (cx + dx * c_len, cy), (6, 182, 212), 3)
        cv2.line(annotated, (cx, cy), (cx, cy + dy * c_len), (6, 182, 212), 3)
    cv2.putText(
        annotated, f"TARGET ROI ({best_score*100:.1f}%)", (bx + 6, max(22, by - 8)),
        cv2.FONT_HERSHEY_SIMPLEX, 0.48, (16, 185, 129), 2, cv2.LINE_AA
    )
    return best_bbox, best_score, annotated

def clean_label_ascii(label_str):
    """กรองเฉพาะชื่อภาษาอังกฤษสำหรับแสดงผลใน Matplotlib ป้องกันอักษรภาษาไทยกลายเป็นกล่องสี่เหลี่ยม"""
    parts = label_str.split('_')
    en_parts = []
    for p in parts:
        # เก็บเฉพาะคำที่มีตัวอักษรภาษาอังกฤษ
        if any('a' <= c.lower() <= 'z' for c in p):
            en_p = ''.join([c for c in p if 'a' <= c.lower() <= 'z' or c.isdigit() or c == '-'])
            if en_p:
                en_parts.append(en_p)
    res = '_'.join(en_parts) if en_parts else 'class'
    return res[:22]

if show_sample_preview and len(classes) > 0:
    print("\n" + "=" * 75)
    print("🖼️ แสดงภาพตัวอย่างชุดข้อมูลจริง พร้อมการตีกรอบตรวจจับวัตถุเป้าหมาย (Intelligent ROI Detection)")
    print("=" * 75)
    n_classes = len(classes)
    n_cols = max(1, min(samples_per_class, 4))
    fig, axes = plt.subplots(n_classes, n_cols, figsize=(4.2 * n_cols, 3.2 * n_classes), squeeze=False)
    
    for row_idx, cls_name in enumerate(classes):
        cls_dir = os.path.join(f"{target_dir}/train", cls_name)
        img_files = [f for f in sorted(os.listdir(cls_dir)) if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))]
        if not img_files:
            continue
        random.seed(42 + row_idx)
        selected_samples = random.sample(img_files, min(n_cols, len(img_files)))
        while len(selected_samples) < n_cols:
            selected_samples.append(selected_samples[-1])
            
        for col_idx, img_name in enumerate(selected_samples):
            img_path = os.path.join(cls_dir, img_name)
            bgr = cv2.imread(img_path)
            if bgr is None:
                continue
            h, w = bgr.shape[:2]
            if show_roi_detection:
                bbox, score, disp_bgr = detect_agri_roi(bgr)
                badge = f"[ROI: {score*100:.0f}%]"
            else:
                disp_bgr = bgr
                badge = ""
            rgb = cv2.cvtColor(disp_bgr, cv2.COLOR_BGR2RGB)
            ax = axes[row_idx, col_idx]
            ax.imshow(rgb)
            display_title = clean_label_ascii(cls_name)
            ax.set_title(f"Class: {display_title} (#{col_idx+1})\n{w}x{h} px  {badge}", fontsize=10, fontweight="bold", pad=4)
            ax.axis("off")
            
    plt.suptitle("Dataset Real Samples & Intelligent ROI Detection Preview", fontsize=14, fontweight="bold", y=1.002)
    plt.tight_layout()
    plt.show()
    print("✅ แสดงภาพตัวอย่างชุดข้อมูลจริง 100% เรียบร้อยแล้ว!")



In [ ]:
#@title 🔍 (เสริม) ตรวจสอบภาพตัวอย่างเจาะลึกและทดสอบระบบตรวจจับวัตถุ (Interactive Target & Quality Gate Explorer) { display-mode: "form" }
import os
import random
import cv2
import numpy as np
import matplotlib.pyplot as plt

#@markdown **เครื่องมือตรวจสอบคุณภาพภาพถ่ายและทดสอบอัลกอริทึมการตรวจจับวัตถุเป้าหมาย (Two-Stage Precision Agri Vision):**
inspect_mode = "🔀 สุ่มทุกคลาส (Random All Classes)" #@param ["🔀 สุ่มทุกคลาส (Random All Classes)", "🎯 ดูเฉพาะคลาสที่เลือก (Single Class Deep Dive)"]
selected_class_to_inspect = "" #@param {type:"string"}
num_display_samples = 4 #@param [2, 4, 6, 8]
show_cropped_target_roi = True #@param {type:"boolean"}

target_dir = "dataset"
if not os.path.exists(f"{target_dir}/train"):
    print("⚠️ กรุณากดรัน [ขั้นตอนที่ 2] ด้านบนก่อน เพื่อโหลดชุดข้อมูลเข้าสู่ระบบ")
else:
    avail_classes = sorted([d for d in os.listdir(f"{target_dir}/train") if os.path.isdir(os.path.join(f"{target_dir}/train", d)) and not d.startswith(".")])
    print(f"🎯 พบคลาสข้อมูลจริงทั้งหมด: {avail_classes}")
    
    candidate_images = []
    if "สุ่มทุกคลาส" in inspect_mode:
        for c in avail_classes:
            cdir = os.path.join(f"{target_dir}/train", c)
            ims = [os.path.join(cdir, f) for f in os.listdir(cdir) if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))]
            if ims:
                candidate_images.extend([(c, p) for p in ims])
        random.shuffle(candidate_images)
        chosen_pairs = candidate_images[:num_display_samples]
    else:
        target_cls = selected_class_to_inspect.strip() if selected_class_to_inspect.strip() in avail_classes else avail_classes[0]
        cdir = os.path.join(f"{target_dir}/train", target_cls)
        ims = [os.path.join(cdir, f) for f in os.listdir(cdir) if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))]
        random.shuffle(ims)
        chosen_pairs = [(target_cls, p) for p in ims[:num_display_samples]]
    
    if not chosen_pairs:
        print("❌ ไม่พบภาพในคลาสที่เลือก กรุณาตรวจสอบอีกครั้ง")
    else:
        def assess_quality(bgr_img):
            gray = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2GRAY)
            lap_var = cv2.Laplacian(gray, cv2.CV_64F).var()
            glare_pct = (np.sum(gray > 248) / float(gray.size)) * 100.0
            blur_status = "คมชัดดีเยี่ยม (Sharp)" if lap_var >= 80 else "เบลอเล็กน้อย (Soft)" if lap_var >= 40 else "เบลอมาก (Blurry)"
            return lap_var, glare_pct, blur_status
        
        n_samples = len(chosen_pairs)
        n_cols = 2 if show_cropped_target_roi else 1
        fig, axes = plt.subplots(n_samples, n_cols, figsize=(5.0 * n_cols, 3.4 * n_samples), squeeze=False)
        
        print("\n" + "=" * 80)
        print("🔬 [รายงานคุณภาพภาพถ่ายและพิกัดการตรวจจับวัตถุเป้าหมาย (Visual Gate Telemetry)]")
        print("=" * 80)
        
        for idx, (cls_name, img_path) in enumerate(chosen_pairs):
            bgr = cv2.imread(img_path)
            if bgr is None:
                continue
            h, w = bgr.shape[:2]
            lap_var, glare_pct, blur_status = assess_quality(bgr)
            bbox, roi_score, annotated_bgr = detect_agri_roi(bgr)
            bx, by, bw, bh = bbox
            
            rgb_full = cv2.cvtColor(annotated_bgr, cv2.COLOR_BGR2RGB)
            ax_full = axes[idx, 0]
            ax_full.imshow(rgb_full)
            display_title = clean_label_ascii(cls_name)
            ax_full.set_title(f"[{display_title}] Full Frame + ROI Detection\nBox: ({bx},{by}) {bw}x{bh} px | Conf: {roi_score*100:.1f}%", fontsize=9, fontweight="bold")
            ax_full.axis("off")
            
            if show_cropped_target_roi:
                cropped = bgr[by:by+bh, bx:bx+bw]
                if cropped.size > 0:
                    rgb_cropped = cv2.cvtColor(cropped, cv2.COLOR_BGR2RGB)
                else:
                    rgb_cropped = rgb_full
                ax_crop = axes[idx, 1]
                ax_crop.imshow(rgb_cropped)
                ax_crop.set_title(f"Target ROI Surface (Cropped)\nSharpness: {lap_var:.1f} [{blur_status}] | Glare: {glare_pct:.1f}%", fontsize=9, fontweight="bold")
                ax_crop.axis("off")
            
            print(f"📸 ตัวอย่างที่ {idx+1}: คลาส '{cls_name}' | ไฟล์: {os.path.basename(img_path)}")
            print(f"   • พิกัด ROI Bounding Box: ({bx}, {by}, {bw}, {bh}) | ความมั่นใจ: {roi_score*100:.1f}%")
            print(f"   • ความคมชัด (Laplacian Variance): {lap_var:.1f} [{blur_status}] | แสงสะท้อนจ้า: {glare_pct:.2f}%")
            print("-" * 80)
            
        plt.suptitle("Interactive Object Detection & Precision Agri Telemetry Explorer", fontsize=13, fontweight="bold", y=1.005)
        plt.tight_layout()
        plt.show()



In [ ]:
#@title ⚙️ ขั้นตอนที่ 3: ตั้งค่า เริ่มการฝึกสอน และประเมินผลโมเดลเชิงลึก (Deep Evaluation Suite) { display-mode: "form" }
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import time
import os

#@markdown **การตั้งค่าพารามิเตอร์แบบ Low-Code:**
model_type = "MobileNetV3 Small (ขนาดเบา เหมาะกับสมาร์ทโฟนและเว็บ)" #@param ["MobileNetV3 Small (ขนาดเบา เหมาะกับสมาร์ทโฟนและเว็บ)", "ResNet18 (โมเดลมาตรฐาน)"]
epochs = 8 #@param {type:"slider", min:3, max:20, step:1}
batch_size = 16 #@param [8, 16, 32]
learning_rate = 0.001 #@param [0.0005, 0.001, 0.005]
enable_augmentation = True #@param {type:"boolean"}

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"🚀 สภาพแวดล้อมประมวลผล: {device} | สถาปัตยกรรม: {model_type}")
print(f"📊 จำนวนรอบ (Epochs): {epochs} | Batch Size: {batch_size} | Learning Rate: {learning_rate}")
print("=" * 75)

# 1. Data Transforms
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip() if enable_augmentation else transforms.Lambda(lambda x: x),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_dataset = datasets.ImageFolder("dataset/train", transform=train_transforms)
val_dataset = datasets.ImageFolder("dataset/val", transform=val_transforms)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

num_classes = len(train_dataset.classes)
class_names = train_dataset.classes
print(f"📸 ชุดข้อมูล: Train = {len(train_dataset)} ภาพ | Validation = {len(val_dataset)} ภาพ | จำนวนคลาส = {num_classes}")
print(f"🎯 รายชื่อคลาส: {class_names}")
print("-" * 75)

# 2. Build Model
if "ResNet" in model_type:
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)
else:
    model = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT)
    in_features = model.classifier[3].in_features
    model.classifier[3] = nn.Linear(in_features, num_classes)
model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=learning_rate)

history_train_loss = []
history_val_loss = []
history_train_acc = []
history_val_acc = []
best_acc = 0.0

# 3. Training Loop with Live Metrics
print()
print("🧠 [ระหว่างเทรนโมเดล] ติดตามความก้าวหน้าแบบ Real-Time:")
print("+" + "-"*10 + "+" + "-"*14 + "+" + "-"*13 + "+" + "-"*13 + "+" + "-"*12 + "+" + "-"*14 + "+")
print("| Epoch    | Train Loss   | Train Acc   | Val Loss    | Val Acc    | สถานะโมเดล   |")
print("+" + "-"*10 + "+" + "-"*14 + "+" + "-"*13 + "+" + "-"*13 + "+" + "-"*12 + "+" + "-"*14 + "+")

start_time = time.time()
for epoch in range(1, epochs + 1):
    ep_start = time.time()
    model.train()
    train_loss = 0.0
    train_correct = 0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * inputs.size(0)
        _, preds = torch.max(outputs, 1)
        train_correct += torch.sum(preds == labels.data)
    
    ep_train_loss = train_loss / len(train_dataset)
    ep_train_acc = (train_correct.double() / len(train_dataset)).item() * 100.0
    
    # Validation Phase
    model.eval()
    val_loss = 0.0
    val_correct = 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            val_correct += torch.sum(preds == labels.data)
            
    ep_val_loss = val_loss / len(val_dataset)
    ep_val_acc = (val_correct.double() / len(val_dataset)).item() * 100.0
    
    history_train_loss.append(ep_train_loss)
    history_val_loss.append(ep_val_loss)
    history_train_acc.append(ep_train_acc)
    history_val_acc.append(ep_val_acc)
    
    status = "            "
    if ep_val_acc > best_acc:
        best_acc = ep_val_acc
        torch.save({'state_dict': model.state_dict(), 'classes': class_names}, "best_agri_model.pt")
        status = "⭐ BEST SAVED"
        
    print(f"| {epoch:02d}/{epochs:02d}    | {ep_train_loss:10.4f}   | {ep_train_acc:9.2f}%  | {ep_val_loss:9.4f}   | {ep_val_acc:8.2f}%  | {status:12s} |")

print("+" + "-"*10 + "+" + "-"*14 + "+" + "-"*13 + "+" + "-"*13 + "+" + "-"*12 + "+" + "-"*14 + "+")
elapsed = time.time() - start_time
print()
print(f"🎉 การฝึกสอนเสร็จสมบูรณ์ 100%! (ใช้เวลารวม {elapsed:.1f} วินาที | ความแม่นยำสูงสุด: {best_acc:.2f}%)")

# 4. Deep Evaluation: Confusion Matrix & Classification Report
print()
print("=" * 75)
print("📊 [เมื่อเทรนเสร็จแล้ว] รายงานและสเปกของโมเดลอย่างละเอียด (Model Evaluation Suite)")
print("="*75)

# โหลด Best Checkpoint สำหรับการประเมิน
checkpoint = torch.load("best_agri_model.pt", map_location=device)
model.load_state_dict(checkpoint["state_dict"])
model.eval()

all_preds = []
all_labels = []
with torch.no_grad():
    for inputs, labels in val_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

# 4.1 ตาราง Cross-Tabulation / Confusion Matrix
cm = np.zeros((num_classes, num_classes), dtype=int)
for t, p in zip(all_labels, all_preds):
    cm[t, p] += 1

df_cm = pd.DataFrame(
    cm, 
    index=[f"ความจริง: {c}" for c in class_names], 
    columns=[f"ทำนาย: {c}" for c in class_names]
)
print()
print("1️⃣ ตาราง Cross-Tabulation (Confusion Matrix Matrix / ตารางไขว้การทำนาย):")
print(df_cm.to_string())

# 4.2 ตาราง Classification Report (Precision, Recall, F1-Score)
metrics_list = []
for i, cls in enumerate(class_names):
    tp = cm[i, i]
    fp = cm[:, i].sum() - tp
    fn = cm[i, :].sum() - tp
    prec = (tp / (tp + fp)) * 100 if (tp + fp) > 0 else 0.0
    rec = (tp / (tp + fn)) * 100 if (tp + fn) > 0 else 0.0
    f1 = (2 * prec * rec) / (prec + rec) if (prec + rec) > 0 else 0.0
    support = cm[i, :].sum()
    metrics_list.append({
        "คลาส (Class)": cls,
        "Precision (%)": f"{prec:.1f}%",
        "Recall (%)": f"{rec:.1f}%",
        "F1-Score (%)": f"{f1:.1f}%",
        "Support (ภาพ)": support
    })
df_report = pd.DataFrame(metrics_list)
print()
print("2️⃣ ตารางวัดประสิทธิภาพรายคลาส (Precision / Recall / F1-Score):")
print(df_report.to_string(index=False))

# 4.3 วัดความเร็ว Inference Latency (ms)
dummy_input = torch.randn(1, 3, 224, 224).to(device)
for _ in range(5): _ = model(dummy_input) # warmup
t_bench_start = time.time()
bench_runs = 50
with torch.no_grad():
    for _ in range(bench_runs):
        _ = model(dummy_input)
avg_latency_ms = ((time.time() - t_bench_start) / bench_runs) * 1000.0

# 4.4 สรุปสเปกโมเดล (Model Specifications Summary Card)
print()
print("3️⃣ สรุปรายละเอียดสเปกโมเดล AI (Model Specifications Card):")
print(f"  • โครงสร้างหลัก (Architecture): {model_type}")
print(f"  • จำนวนพารามิเตอร์ทั้งหมด (Total Parameters): {total_params:,} พารามิเตอร์")
print(f"  • พารามิเตอร์ที่ฝึกสอนได้ (Trainable Parameters): {trainable_params:,}")
print(f"  • ความเร็วในการพยากรณ์ผล (Inference Latency): {avg_latency_ms:.2f} ms ต่อภาพ (บน {device})")
print(f"  • ขนาดไฟล์โมเดลโดยประมาณ (Model Weight Size): ~{os.path.getsize('best_agri_model.pt') / (1024*1024):.2f} MB (เหมาะกับ Edge AI & เว็บ)")
print(f"  • มิติภาพนำเข้า (Input Tensor Shape): (Batch, 3, 224, 224) RGB Normalized")
print(f"  • ความแม่นยำสูงสุด (Best Top-1 Validation Accuracy): {best_acc:.2f}%")

# 5. Dual Performance Curves & Heatmap Visualizations
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))

# Graph 1: Loss Curve
ax1.plot(range(1, epochs + 1), history_train_loss, color="#f59e0b", linewidth=2, marker="o", label="Train Loss")
ax1.plot(range(1, epochs + 1), history_val_loss, color="#ef4444", linewidth=2, marker="s", label="Val Loss")
ax1.set_title("Learning Loss Curve (Overfitting Check)", fontsize=12, fontweight="bold")
ax1.set_xlabel("Epochs")
ax1.set_ylabel("Loss")
ax1.grid(True, linestyle="--", alpha=0.5)
ax1.legend()

# Graph 2: Accuracy Curve
ax2.plot(range(1, epochs + 1), history_train_acc, color="#3b82f6", linewidth=2, marker="o", label="Train Accuracy")
ax2.plot(range(1, epochs + 1), history_val_acc, color="#10b981", linewidth=2, marker="^", label="Val Accuracy")
ax2.set_title("Accuracy Progression Curve (%)", fontsize=12, fontweight="bold")
ax2.set_xlabel("Epochs")
ax2.set_ylabel("Accuracy (%)")
ax2.grid(True, linestyle="--", alpha=0.5)
ax2.legend()

# Graph 3: Confusion Matrix Heatmap
im = ax3.imshow(cm, interpolation="nearest", cmap=plt.cm.Greens)
ax3.set_title("Confusion Matrix Heatmap (Cross-Table)", fontsize=12, fontweight="bold")
tick_marks = np.arange(len(class_names))
short_names = [c[:12] + ".." if len(c) > 12 else c for c in class_names]
ax3.set_xticks(tick_marks)
ax3.set_yticks(tick_marks)
ax3.set_xticklabels(short_names, rotation=45, ha="right")
ax3.set_yticklabels(short_names)
thresh = cm.max() / 2.0
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax3.text(j, i, format(cm[i, j], "d"),
                 ha="center", va="center",
                 color="white" if cm[i, j] > thresh else "black",
                 fontweight="bold")
ax3.set_ylabel("True Class (ความจริง)")
ax3.set_xlabel("Predicted Class (ผลทำนาย)")
plt.colorbar(im, ax=ax3, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()


In [ ]:
#@title ⚡ ขั้นตอนที่ 4: ส่งออกโมเดลมาตรฐานเปิด (ONNX) และดาวน์โหลดลงเครื่อง { display-mode: "form" }
import torch

#@markdown คลิกปุ่ม Play [ ▶ ] เพื่อส่งออกและดาวน์โหลดไฟล์โมเดลไปใช้งานในแอปพลิเคชัน:
checkpoint = torch.load("best_agri_model.pt", map_location=device)
model.load_state_dict(checkpoint["state_dict"])
model.eval()

dummy_input = torch.randn(1, 3, 224, 224).to(device)
onnx_path = "smart_agri_model.onnx"

torch.onnx.export(
    model, dummy_input, onnx_path,
    input_names=["input"], output_names=["output"],
    dynamic_axes={"input": {0: "batch_size"}, "output": {0: "batch_size"}}
)
print(f"✅ แปลงโมเดลเป็นมาตรฐานเปิด ONNX สำเร็จ: {onnx_path} (ขนาดไฟล์ ~8.5 MB)")

try:
    from google.colab import files
    print("📥 กำลังดาวน์โหลดไฟล์โมเดลลงสู่คอมพิวเตอร์ของคุณอัตโนมัติ...")
    files.download(onnx_path)
except Exception:
    print(f"💾 ไฟล์โมเดลถูกบันทึกไว้ที่: {os.path.abspath(onnx_path)}")


---
### 🎯 สิ่งที่ต้องทำต่อไปใน Workshop:
1. นำไฟล์  ที่ดาวน์โหลดได้ ไปเปิดใน **Streamlit Web Dashboard (แท็บที่ 1 หรือ 3)**
2. ถ่ายรูปใบพืชหรือดินจริง แล้วทดสอบการวินิจฉัยทันที!

*สาขาวิชาปัญญาประดิษฐ์เพื่อเกษตรดิจิทัลและสิ่งแวดล้อม คณะวิทยาศาสตร์และเทคโนโลยี มหาวิทยาลัยราชภัฏรำไพพรรณี*